In [284]:
import yfinance as yf
import matplotlib.pyplot as plt

In [285]:
import datetime as dt

In [286]:
dt.datetime(2026, 5, 2) - dt.timedelta(days=730)

datetime.datetime(2024, 5, 2, 0, 0)

In [287]:
data_nvda = yf.Ticker("NVDA").history(start='2024-05-10', end='2026-05-02', interval="1h")
data_appl = yf.Ticker("AAPL").history(start='2024-05-10', end='2026-05-02', interval="1h")
data_msft = yf.Ticker("MSFT").history(start='2024-05-10', end='2026-05-02', interval="1h")
data_BTC = yf.Ticker("BTC-USD").history(start='2024-05-10', end='2026-05-02', interval="1h")
data_ETH = yf.Ticker("ETH-USD").history(start='2024-05-10', end='2026-05-02', interval="1h")
data_sp500 = yf.Ticker("SP500-USD").history(start='2024-05-10', end='2026-05-02', interval="1h")
data_nasqad = yf.Ticker("^IXIC").history(start='2024-05-10', end='2026-05-02', interval="1h")

In [288]:
import pandas as pd

In [289]:
start = pd.Timestamp('2024-05-10 09:30:00').tz_localize('America/New_York')
end = pd.Timestamp('2024-05-11 00:00:00').tz_localize('America/New_York')

In [ ]:
def _load_data(start_date: str, end_date: str):
    """Загружает OHLCV для всех тикеров через yfinance, возвращает плоский DataFrame."""
    # Скачиваем все тикеры одним запросом (быстрее и удобнее)
    data = yf.download(
        ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',],
        start=start_date,
        end=end_date,
        group_by='ticker',   # <- важный параметр: возвращает MultiIndex (Ticker, Price)
        progress=False,
        auto_adjust=False    # оставляем оригинальные OHLCV
    )

    df_list = []
    for ticker in ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',]:
        ticker_df = data[ticker].copy()
        ticker_df.columns = [f"{ticker}_{col}" for col in ticker_df.columns]
        df_list.append(ticker_df)
    df = pd.concat(df_list, axis=1, join='inner')

    # Убедимся, что индекс — это datetime
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)

    # Проверка: все ли ожидаемые колонки присутствуют
    expected_cols = [f"{ticker}_{field}" for ticker in ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',] for field in ['Open','High','Low','Close','Volume']]
    missing = set(expected_cols) - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns after download: {missing}")

    return df, df.values.astype(np.float32), df.index

In [ ]:
def calculate_features(data):
    n = len(self.dates)
    n_assets = self.n_assets
    n_features = 8
    features = np.zeros((n, n_features, n_assets), dtype=np.float32)

    for i, ticker in enumerate(self.tickers):
        close = self.prices[f"{ticker}_Close"].values.astype(float)
        high = self.prices[f"{ticker}_High"].values.astype(float)
        low = self.prices[f"{ticker}_Low"].values.astype(float)
        volume = self.prices[f"{ticker}_Volume"].values.astype(float)

        # Защита от нулевых и отрицательных цен
        close = np.maximum(close, 1e-6)
        high = np.maximum(high, close)
        low = np.minimum(low, close)

        # 1. Лог доходность 1 день
        ret1 = np.diff(np.log(close), prepend=np.log(close[0]))
        # 2. Лог доходность 5 дней
        ret5 = np.log(close) - np.log(np.roll(close, 5))
        ret5[:5] = 0.0
        # 3. RSI(14)
        delta = np.diff(close, prepend=close[0])
        gain = np.clip(delta, 0, None)
        loss = np.clip(-delta, 0, None)
        avg_gain = self._sma(gain, 14)
        avg_loss = self._sma(loss, 14)
        rs = np.divide(avg_gain, avg_loss + 1e-8)
        rsi = 100 - 100 / (1 + rs)
        # 4. Bollinger %B
        sma20 = self._sma(close, 20)
        std20 = self._rolling_std(close, 20) + 1e-8
        bb_pct_b = (close - (sma20 - 2*std20)) / (4*std20 + 1e-8)
        # 5. ATR / close
        tr = np.maximum(high - low,
                        np.abs(high - np.roll(close, 1)),
                        np.abs(low - np.roll(close, 1)))
        atr = self._sma(tr, 14)
        atr_rel = atr / (close + 1e-8)
        # 6. Объемный импульс
        vol_ma20 = self._sma(volume, 20)
        vol_ratio = volume / (vol_ma20 + 1e-8)
        # 7. Нормализованная позиция в диапазоне дня
        daily_pos = (close - low) / (high - low + 1e-8)
        # 8. Моментум 5 дней
        mom5 = close / (np.roll(close, 5) + 1e-8) - 1
        mom5[:5] = 0.0

        # Заполняем массив
        features[:, 0, i] = np.nan_to_num(ret1, nan=0.0, posinf=1.0, neginf=-1.0)
        features[:, 1, i] = np.nan_to_num(ret5, nan=0.0, posinf=1.0, neginf=-1.0)
        features[:, 2, i] = np.nan_to_num(rsi / 100.0, nan=0.5, posinf=1.0, neginf=0.0)
        features[:, 3, i] = np.nan_to_num(bb_pct_b, nan=0.5, posinf=1.0, neginf=0.0)
        features[:, 4, i] = np.nan_to_num(atr_rel, nan=0.0, posinf=1.0, neginf=0.0)
        features[:, 5, i] = np.nan_to_num(vol_ratio, nan=1.0, posinf=10.0, neginf=0.0)
        features[:, 6, i] = np.nan_to_num(daily_pos, nan=0.5, posinf=1.0, neginf=0.0)
        features[:, 7, i] = np.nan_to_num(mom5, nan=0.0, posinf=1.0, neginf=-1.0)

        # Клиппинг, чтобы не было гигантских выбросов
        features[:, :, i] = np.clip(features[:, :, i], -5.0, 5.0)

    # Z‑нормализация по скользящему окну
    norm_features = np.zeros_like(features)
    for t in range(self.lookback, n):
        window = features[t-self.lookback:t, :, :]
        mean = np.mean(window, axis=0, keepdims=True)
        std = np.std(window, axis=0, keepdims=True)
        std = np.where(std < 1e-6, 1.0, std)   # избегаем деления на ноль
        norm_features[t] = (features[t] - mean) / std
    norm_features[:self.lookback] = 0.0
    self.features_array = norm_features.astype(np.float32)